# Goodput Colab GPU demo (ticket 4.2)

Short **single-process** train on **synthetic tensors** using the same `train_from_settings` path as CI. This is **not** a multi-node cluster and **not** a dataset download.

## Settings (must match `experiments/colab.yaml`)

| Knob | Value |
|------|-------|
| `seed` | 42 |
| `steps` | 12 |
| `batch_size` | 8 |
| `input_size` | 16 |
| `hidden_size` | 32 |
| `learning_rate` | 0.01 |
| `ckpt_interval` | 4 |
| `ckpt_mode` | naive |
| `num_workers` | 1 |
| `device` | `cuda` if a GPU is attached, else CPU |

## Colab steps

1. **Runtime → Change runtime type → T4 GPU** (or CPU if the GPU quota is empty).
2. Run all cells.
3. Expected: finite loss, `goodput` in `[0, 1]`, a `report.json` under `artifacts/reports/colab-gpu-demo/`.

Local equivalent: `goodput-run --config experiments/colab.yaml` (falls back to CPU without CUDA).

In [ ]:
# Install from GitHub (public repo). CPU torch from PyPI is enough if you skip the GPU runtime.
import subprocess, sys
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/cletusabumah/goodput.git"]
)

In [ ]:
import torch
from pathlib import Path
from goodput.config import Settings
from goodput.metrics import build_run_report
from goodput.providers import LocalFsCheckpointStore, build_providers
from goodput.training import train_from_settings

# Same knobs as experiments/colab.yaml. CUDA if Colab attached a GPU; else CPU.
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch={torch.__version__} cuda_available={torch.cuda.is_available()} device={device}")
if device == "cuda":
    print("gpu=", torch.cuda.get_device_name(0))

settings = Settings(
    run_name="colab-gpu-demo",
    seed=42,
    num_workers=1,
    steps=12,
    batch_size=8,
    input_size=16,
    hidden_size=32,
    learning_rate=0.01,
    ckpt_interval=4,
    ckpt_mode="naive",
    ckpt_dir=Path("artifacts/checkpoints/colab"),
    device=device,
    artifacts_dir=Path("artifacts"),
    metrics_provider="json_file",
    ci_mode=False,
)
store = LocalFsCheckpointStore(settings.ckpt_dir)
result = train_from_settings(settings, checkpoint_store=store)
print(
    f"ok={result.ok} steps={result.steps_completed} "
    f"final_loss={result.final_loss:.6f} device={result.device}"
)

In [ ]:
from goodput.metrics import emit_run_report

report = build_run_report(
    settings=settings,
    wall_seconds=result.wall_seconds,
    useful_seconds=result.useful_seconds,
    steps_completed=result.steps_completed,
    ckpt_save_seconds=result.ckpt_save_seconds,
    ckpt_restore_seconds=result.ckpt_restore_seconds,
    final_loss=result.final_loss,
    extra={"mode": "colab_demo", "torch_device": result.device},
)
providers = build_providers(settings)
emit_run_report(providers.metrics, report)
print(
    f"goodput={report['goodput']:.4f} ckpt_save_s={report['ckpt_save_s']:.6f} "
    f"ckpt_restore_s={report['ckpt_restore_s']:.6f}"
)
print("report", settings.artifacts_dir / "reports" / settings.run_name / "report.json")
assert result.ok and 0.0 <= report["goodput"] <= 1.0